# GPU vs FPGA — Rate-Distortion Comparison

Compare inference quality of the **ResSHyp-relu** model family on GPU (W&B) and on FPGA (ZCU102) for the sub500 test set.

**Data sources:**
- **GPU metrics** — fetched from the W&B project via the API (`test_sub500/bpp`, `test_sub500/psnr_merlin`, etc.)
- **FPGA metrics** — read from disk (`compiled_models/<name>/results/metrics.json`) after `batch_deploy.py`

**Structure:**
1. Setup & filters
2. Load GPU runs → tidy DataFrame
3. Load FPGA results → tidy DataFrame
4. Merge & coverage check
5. Aggregate statistics across seeds
6. RD-curve plots (GPU vs FPGA)
7. FPGA degradation Δ plot
8. BPP comparison (likelihood vs rANS bitstream)
9. Hamburg tile visualization
10. Save figures & per-λ summary table


## 1 · Setup & filters

In [ ]:
import json
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
import torch
import wandb
from omegaconf import OmegaConf

ROOT_DIR = Path("..").resolve()
sys.path.insert(0, str(ROOT_DIR))

from src.utils.constants import AMP_LIN_99, EPS
from src.utils.metrics import psnr as _src_psnr
from src.utils.processing_utils import clip as clip_logI

# ── W&B project ────────────────────────────────────────────────────────────────
ENTITY = "cedric-leonard"
PROJECT = "SAR_DDC_FPGA"

# ── FPGA compiled models directory ─────────────────────────────────────────────
COMPILED_MODELS_DIR = ROOT_DIR / "results" / "fpga" / "compiled_models"

# ── Run filters (mirrors batch_deploy.py FILTERS_CONFIG) ───────────────────────
# Seeds 0-5 only; seed=42 is reserved for manual tests and excluded everywhere.
ACCEPTED_SEEDS = [0, 1, 2, 3, 4, 5]

FILTERS_CONFIG: List[Tuple] = [
    ("model.net.activation", "==", "relu"),
    ("model.net.no_output_padding", "==", True),
    ("seed", "in", ACCEPTED_SEEDS),
]

# ── Shared color palette (Okabe-Ito, colorblind-safe) ─────────────────────────
C = json.load(open(ROOT_DIR / "notebooks" / "plots_colors.json"))

# ── Plotting style ──────────────────────────────────────────────────────────────
COLORS  = {
    # Per-backend (used in §7 delta, §8 BPP, §9 tile vis — backend color encoding)
    "gpu":  C["compare_gpu_fpga"]["gpu"],    # "#0072B2" blue
    "fpga": C["compare_gpu_fpga"]["fpga"],   # "#009E73" green
    # Per-architecture (used in §6 RD curves — color=arch, linestyle=backend)
    "ResSHyp": C["compare_gpu_fpga"]["gpu"],  # "#0072B2" blue    — with residual blocks
    "SHyp":    C["architectures"]["SHyp"],    # "#E69F00" orange  — without residual blocks
    # Per-metric (used in §7 multi-metric delta)
    "psnr":       C["metrics"]["psnr"],
    "ssim":       C["metrics"]["ssim"],
    "epd":        C["metrics"]["epd"],
    "error_bars": C["metrics"]["error_bars"],
}
MARKERS    = {"gpu": "o", "fpga": "s"}
LINESTYLES = {"gpu": "-", "fpga": "--"}


SAVE_FIGURES = True   # set True to write PNGs to PLOTS_DIR
PLOTS_DIR = ROOT_DIR / "results" / "plots"

# ── ANSI shortcuts ──────────────────────────────────────────────────────────────
r, g, b, y, e = "\033[31m", "\033[32m", "\033[34m", "\033[33m", "\033[0m"


## 2 · Load GPU runs → tidy DataFrame

Fetch all matching runs from W&B. One row per run (one `(lambda, seed)` pair).

**`groupby` primer** — `df.groupby("lambda")` splits the DataFrame into sub-groups by unique λ value,
returning a `GroupBy` object. Nothing is computed yet. `.agg(["mean","std"])` then applies each
aggregation to every column in each group, returning a new DataFrame with a two-level column index
`(metric, stat)`. Access: `stats_df[("bpp_gpu","mean")]` or equivalently `stats_df["bpp_gpu"]["mean"]`.

**Tidy / long format** — one row per `(lambda, seed, backend)`. This is the key design choice:
it makes aggregation, filtering, and plotting one-liners instead of nested loops.


In [ ]:
def _check_condition(value: Any, op: str, test_value: Any) -> bool:
    """Single filter condition — mirrors batch_deploy.py / update_wandb_runs.py."""
    if op == "==":      return value == test_value
    if op == "!=":      return value != test_value
    if op == "in":      return value in test_value
    if op == "not in":  return value not in test_value
    if op == "exists":  return value is not None
    if op == "is_none": return value is None
    if op == ">":       return value is not None and float(value) > float(test_value)
    if op == "<":       return value is not None and float(value) < float(test_value)
    raise ValueError(f"Unsupported op: {op!r}")


def _run_matches(run_cfg: dict) -> bool:
    cfg = OmegaConf.create(run_cfg)
    return all(
        _check_condition(OmegaConf.select(cfg, key), op, val)
        for key, op, val in FILTERS_CONFIG
    )


def load_gpu_runs() -> pd.DataFrame:
    """Fetch matching W&B runs and return a tidy DataFrame (one row per run).

    Columns: lambda, seed, wandb_id, wandb_name, backend,
             bpp_likelihood, bpp_bitstream (NaN if not yet computed),
             psnr_merlin, psnr_adam_noc, mse_merlin, mse_adam_noc,
             ssim_merlin, ssim_adam_noc, ms_ssim_merlin, ms_ssim_adam_noc,
             enl_recon, ratio_mean, ratio_enl, epd_merlin, epd_adam_noc,
             run_dir  ← Hydra output directory (used to locate Hamburg tile .npy files)

    Notes:
    - bpp_bitstream comes from test/bpp_bitstream (added by update_wandb_runs.py once that
      pipeline is complete). Currently NaN for all runs — see cell 8 for the BPP comparison.
    - MERLIN_DDS metrics are included if present (they were added in a later evaluation pass).
    """
    api = wandb.Api()
    print(f"{b}Fetching runs from {ENTITY}/{PROJECT}...{e}")
    all_runs = list(api.runs(f"{ENTITY}/{PROJECT}"))
    matching = [r for r in all_runs if _run_matches(r.config)]
    print(f"  {len(all_runs)} total  →  {g}{len(matching)} match filters{e}")

    rows = []
    for run in matching:
        s = run.summary._json_dict
        # Extract Hydra run directory from config (nested under "paths")
        paths_cfg = run.config.get("paths", {})
        run_dir = paths_cfg.get("output_dir", "") if isinstance(paths_cfg, dict) else ""
        row: Dict[str, Any] = {
            "lambda":          run.config.get("lambda"),
            "seed":            run.config.get("seed"),
            "wandb_id":        run.id,
            "wandb_name":      run.name,
            "backend":         "gpu",
            "arch":            "ResSHyp" if "ResSHyp" in (run.name or "") else ("SHyp" if "SHyp" in (run.name or "") else "unknown"),
            # Path to the Hydra run directory (for Hamburg tile .npy / metrics.json lookups)
            "run_dir":         run_dir,
            # BPP — likelihood (always present) and bitstream (added later by update_wandb_runs.py)
            "bpp_likelihood":  s.get("test_sub500/bpp"),
            "bpp_bitstream":   s.get("test_sub500/bpp_bitstream"),  # NaN until update_wandb_runs finishes
            # Quality vs MERLIN reference
            "psnr_merlin":     s.get("test_sub500/psnr_merlin"),
            "mse_merlin":      s.get("test_sub500/mse_merlin"),
            "ssim_merlin":     s.get("test_sub500/ssim_merlin"),
            "ms_ssim_merlin":  s.get("test_sub500/ms_ssim_merlin"),
            # Quality vs ADAM-NOC reference
            "psnr_adam_noc":   s.get("test_sub500/psnr_adam_noc"),
            "mse_adam_noc":    s.get("test_sub500/mse_adam_noc"),
            "ssim_adam_noc":   s.get("test_sub500/ssim_adam_noc"),
            "ms_ssim_adam_noc":s.get("test_sub500/ms_ssim_adam_noc"),
            # SAR quality metrics (reference-free or noisy-paired)
            "enl_recon":       s.get("test_sub500/enl_recon"),
            "ratio_mean":      s.get("test_sub500/ratio_mean"),
            "ratio_enl":       s.get("test_sub500/ratio_enl"),
            "epd_merlin":      s.get("test_sub500/epd_merlin"),
            "epd_adam_noc":    s.get("test_sub500/epd_adam_noc"),
        }
        rows.append(row)

    df = pd.DataFrame(rows).sort_values(["lambda", "seed"]).reset_index(drop=True)
    print(f"  Loaded {g}{len(df)}{e} GPU runs  ({df['lambda'].nunique()} λ × {df['seed'].nunique()} seeds)")
    return df


gpu_df = load_gpu_runs()
gpu_df.head(6)


## 3 · Load FPGA results → tidy DataFrame

Scan `compiled_models/` on disk. Each model directory is linked back to a W&B run via
`manifest.json` — first by `wandb_run_id` (present on runs deployed after the fix to
`deploy.py` today), then by `(seed, lambda)` lookup for the 60 existing runs.

**FPGA metrics available:** `bpp` (real rANS bitstream), `psnr` vs Noisy / ADAM / MERLIN.
No SSIM on FPGA yet — those columns are `NaN`. For a future implementation in pure NumPy,
see the comment at the bottom of this cell.


In [ ]:
def _build_wandb_id_lookup(gpu_df: pd.DataFrame) -> Dict[Tuple, str]:
    """Build a (seed, lambda) -> wandb_id lookup from the already-loaded GPU DataFrame.

    Used as a fallback for manifests that predate the wandb_run_id field in manifest.json
    (i.e. the 60 runs deployed before today's deploy.py fix).
    The key (seed, lambda) is unique within the relu/no_output_padding filter set.
    """
    return {(row.seed, row["lambda"]): row.wandb_id for _, row in gpu_df.iterrows()}


def load_fpga_results(gpu_df: pd.DataFrame) -> pd.DataFrame:
    """Scan compiled_models/ and return a tidy DataFrame (one row per model with results).

    Filters: seed must be in ACCEPTED_SEEDS (excludes seed=42 and other manual runs).
    Links each FPGA result to a W&B run ID via manifest.json wandb_run_id if available,
    otherwise via (seed, lambda) lookup from gpu_df.

    FPGA metrics schema (from results/metrics.json):
        {"Noisy":  {"bpp": ..., "mse": ..., "psnr": ..., "ssim": ..., "enl": ...,
                    "ratio_mean": ..., "ratio_enl": ...},
         "ADAM":   {"bpp": ..., "mse": ..., "psnr": ..., "ssim": ..., "epd": ...},
         "MERLIN": {"bpp": ..., "mse": ..., "psnr": ..., "ssim": ..., "epd": ...},
         "recon":  {"enl": ..., "ratio_mean": ..., "ratio_enl": ...}}
    bpp is the same for all references (it's a property of the bitstream, not the reference).

    Note on SSIM/MS-SSIM:
        Not currently computed on FPGA because torchmetrics is unavailable there.
        Both metrics can be implemented in pure NumPy (scipy.ndimage.gaussian_filter for
        the local statistics) to within ~0.005 of torchmetrics values. Worth adding to
        inference_utils.py in a future pass.
    """
    wandb_lookup = _build_wandb_id_lookup(gpu_df)
    rows = []
    skipped = []

    for model_dir in sorted(COMPILED_MODELS_DIR.iterdir()):
        manifest_path = model_dir / "manifest.json"
        metrics_path  = model_dir / "results" / "metrics.json"

        if not manifest_path.exists():
            continue
        manifest = json.loads(manifest_path.read_text())

        seed   = manifest.get("seed")
        lmbda  = manifest.get("lambda")

        # Exclude seeds outside the accepted set (e.g. seed=42 manual runs)
        if seed not in ACCEPTED_SEEDS:
            skipped.append(model_dir.name)
            continue

        if not metrics_path.exists():
            print(f"  {y}WARNING{e}: no results/metrics.json for {model_dir.name} — skipping")
            continue

        metrics = json.loads(metrics_path.read_text())

        # Resolve W&B run ID: prefer manifest field (new deploys), fall back to lookup
        wandb_id = manifest.get("wandb_run_id") or wandb_lookup.get((seed, lmbda), "")

        row: Dict[str, Any] = {
            "lambda":          lmbda,
            "seed":            seed,
            "wandb_id":        wandb_id,
            "wandb_name":      manifest.get("model_name", model_dir.name),
            "backend":         "fpga",
            "arch":            "ResSHyp" if "ResSHyp" in manifest.get("model_name", model_dir.name) else ("SHyp" if "SHyp" in manifest.get("model_name", model_dir.name) else "unknown"),
            # Path to the compiled model directory (for Hamburg tile .npy / metrics.json lookups)
            "model_dir":       str(model_dir),
            # BPP from real rANS bitstream (all references share the same bpp)
            "bpp_likelihood":  float("nan"),  # not applicable for FPGA
            "bpp_bitstream":   metrics.get("MERLIN", {}).get("bpp"),
            # Quality vs MERLIN reference
            "psnr_merlin":     metrics.get("MERLIN", {}).get("psnr"),
            "mse_merlin":      metrics.get("MERLIN", {}).get("mse"),
            "ssim_merlin":     metrics.get("MERLIN", {}).get("ssim"), 
            "ms_ssim_merlin":  float("nan"),  # Cannot be computed on FPGA
            # Quality vs ADAM-NOC reference
            "psnr_adam_noc":   metrics.get("ADAM", {}).get("psnr"),
            "mse_adam_noc":    metrics.get("ADAM", {}).get("mse"),
            "ssim_adam_noc":   metrics.get("ADAM", {}).get("ssim"),
            "ms_ssim_adam_noc":float("nan"),   # Cannot be computed on FPGA
            # SAR quality metrics — from the "recon" key (reference-free)
            "enl_recon":       metrics.get("recon", {}).get("enl"),
            "ratio_mean":      metrics.get("recon", {}).get("ratio_mean"),
            "ratio_enl":       metrics.get("recon", {}).get("ratio_enl"),
            "epd_merlin":      metrics.get("MERLIN", {}).get("epd"),
            "epd_adam_noc":    metrics.get("ADAM", {}).get("epd"),
        }
        rows.append(row)

    df = pd.DataFrame(rows).sort_values(["lambda", "seed"]).reset_index(drop=True)
    if skipped:
        print(f"  Skipped {len(skipped)} models outside ACCEPTED_SEEDS: {skipped}")
    print(f"  Loaded {g}{len(df)}{e} FPGA results  ({df['lambda'].nunique()} λ × {df['seed'].nunique()} seeds)")
    no_link = df[df["wandb_id"] == ""]
    if len(no_link) > 0:
        print(f"  {y}WARNING{e}: {len(no_link)} FPGA results could not be linked to a W&B run ID:")
        for _, row in no_link.iterrows():
            print(f"    λ={row['lambda']} seed={row['seed']}  ({row['wandb_name']})")
    return df


fpga_df = load_fpga_results(gpu_df)
fpga_df.head(6)


## 4 · Merge & coverage check

Concatenate GPU and FPGA rows into one DataFrame and verify we have both backends for
every `(lambda, seed)` pair. Missing combinations are shown explicitly.

**`pivot_table` primer** — reshapes long → wide: `index` defines rows, `columns` defines
the new column headers (here `backend`), `values` is the cell content. Think of it as a
spreadsheet cross-tab. Missing combinations appear as `NaN`.


In [ ]:
# Merge both backends into a single tidy DataFrame
df = pd.concat([gpu_df, fpga_df], ignore_index=True)
df["lambda"] = df["lambda"].astype(float)
df["seed"]   = df["seed"].astype(int)
df = df.sort_values(["lambda", "seed", "backend"]).reset_index(drop=True)

print(f"Full DataFrame: {len(df)} rows ({df['backend'].value_counts().to_dict()})")
print(f"λ values: {sorted(df['lambda'].unique())}")
print(f"Seeds:    {sorted(df['seed'].unique())}")

# ── Coverage table ────────────────────────────────────────────────────────────
# pivot_table: rows=lambda, columns=backend, cells=count of runs present (should all be 6)
coverage = df.pivot_table(
    index=["lambda", "arch"], columns="backend", values="psnr_merlin", aggfunc="count"
).rename_axis(None, axis=1)

missing_gpu  = coverage[coverage["gpu"]  < len(ACCEPTED_SEEDS)] if "gpu"  in coverage else []
missing_fpga = coverage[coverage["fpga"] < len(ACCEPTED_SEEDS)] if "fpga" in coverage else []

print(f"\nCoverage (expected {len(ACCEPTED_SEEDS)} runs per λ per backend):")
print(coverage.to_string())
if len(missing_gpu):
    print(f"\n{y}WARNING: incomplete GPU coverage:{e}")
    print(missing_gpu)
if len(missing_fpga):
    print(f"\n{y}WARNING: incomplete FPGA coverage:{e}")
    print(missing_fpga)
else:
    print(f"\n{g}✓ Full coverage: all (λ, seed) pairs present on both backends.{e}")


## 5 · Aggregate statistics across seeds

`groupby(["lambda","backend"]).agg(...)` produces a DataFrame indexed by `(lambda, backend)` with a
two-level column `(metric, stat)`. This is the single source of truth for all plots below.

`query("backend == 'gpu'")` — pandas SQL-style row filter. Returns a view (not a copy), so it's
fast and readable. Equivalent to `df[df["backend"] == "gpu"]` but cleaner for compound conditions.


In [ ]:
METRIC_COLS = [
    "bpp_likelihood", "bpp_bitstream",
    "psnr_merlin", "mse_merlin", "ssim_merlin", "ms_ssim_merlin",
    "psnr_adam_noc", "mse_adam_noc", "ssim_adam_noc", "ms_ssim_adam_noc",
    # SAR quality metrics
    "enl_recon", "ratio_mean", "ratio_enl",
    "epd_merlin", "epd_adam_noc",
]

# groupby + agg: for each (lambda, backend) group compute mean/std/min/max of every metric.
# Result is a DataFrame with MultiIndex columns: (metric_col, stat) e.g. ("psnr_merlin","mean").
stats_df = (
    df.groupby(["lambda", "backend", "arch"])[METRIC_COLS]
    .agg(["mean", "std", "min", "max"])
    .sort_index()  # sort by (lambda, backend, arch)
)

# ── BPP column selection ──────────────────────────────────────────────────────
# Use bpp_bitstream if ALL runs (both backends) have it; else fall back to
# bpp_likelihood for GPU and bpp_bitstream for FPGA (they are not directly comparable
# but are the best available proxies per backend).
gpu_has_bitstream = df.query("backend == 'gpu'")["bpp_bitstream"].notna().all()
if gpu_has_bitstream:
    BPP_GPU_COL = "bpp_bitstream"
    print(f"{g}✓ All GPU runs have bpp_bitstream — using it for RD-curves.{e}")
else:
    BPP_GPU_COL = "bpp_likelihood"
    n_missing = df.query("backend == 'gpu'")["bpp_bitstream"].isna().sum()
    print(f"{y}⚠  {n_missing} GPU runs missing bpp_bitstream — using bpp_likelihood (soft entropy estimate).{e}")
    print("   Run update_wandb_runs.py to compute real bitstream BPP.")

BPP_FPGA_COL = "bpp_bitstream"  # always real rANS on FPGA
print(f"   GPU  BPP column : {BPP_GPU_COL}")
print(f"   FPGA BPP column : {BPP_FPGA_COL}")

# Quick preview: mean PSNR vs MERLIN at each λ for each backend
preview = stats_df["psnr_merlin"]["mean"].unstack(["arch", "backend"])
n_seeds = df["seed"].nunique()
print(f"\nMean PSNR vs MERLIN across {n_seeds} seeds:")
print(preview.to_string(float_format="{:.2f}".format))


## 6 · RD-curve plots — GPU vs FPGA

One curve per backend. Error band = min/max range across seeds. Error bars = ±1 std on BPP.

The X-axis uses `BPP_GPU_COL` for GPU (likelihood until `update_wandb_runs.py` finishes,
then switches automatically to bitstream) and always real rANS BPP for FPGA.
When both are bitstream BPP the curves are on a directly comparable x-axis.


In [ ]:
def plot_rd_curve(
    ax: plt.Axes,
    stats: pd.DataFrame,
    backend: str,
    arch: str,
    bpp_col: str,
    quality_col: str,
    label: Optional[str] = None,
    annotate_lambda: bool = False,
) -> None:
    """Plot a single RD curve (one backend + architecture) with error bands and BPP error bars.

    Args:
        stats:        stats_df already filtered to one (backend, arch) slice (index = lambda).
        backend:      "gpu" or "fpga" — controls linestyle and marker.
        arch:         "ResSHyp" or "SHyp" — controls color.
        bpp_col:      which BPP column to use on X axis.
        quality_col:  DataFrame column key string, e.g. "psnr_merlin".
        annotate_lambda: if True, label each point with its λ value.
    """
    bpp_mean  = stats[(bpp_col, "mean")].astype(float)
    bpp_std   = stats[(bpp_col, "std")].astype(float).fillna(0)
    q_mean    = stats[(quality_col, "mean")].astype(float)
    q_std     = stats[(quality_col, "std")].astype(float)
    q_min     = stats[(quality_col, "min")].astype(float)
    q_max     = stats[(quality_col, "max")].astype(float)

    # Sort by bpp for connected line
    order     = bpp_mean.argsort()
    bpp_s     = bpp_mean.iloc[order].values
    q_s       = q_mean.iloc[order].values
    bpp_std_s = bpp_std.iloc[order].values
    lambdas   = stats.index.get_level_values("lambda").to_numpy()[order]

    color  = COLORS.get(arch, COLORS[backend])
    marker = MARKERS[backend]
    ls     = LINESTYLES[backend]
    lbl    = label or f"{arch} {backend.upper()}"

    ax.plot(bpp_s, q_s, marker=marker, linestyle=ls, color=color, label=lbl, linewidth=1.5, markersize=5)
    ax.fill_between(bpp_s, q_mean.iloc[order].values - q_std.iloc[order].values,
                           q_mean.iloc[order].values + q_std.iloc[order].values,
                    alpha=0.15, color=color)
    ax.errorbar(bpp_s, q_s, xerr=bpp_std_s, fmt="none", ecolor=COLORS["error_bars"], alpha=0.5, capsize=2)

    if annotate_lambda:
        for x, y, lam in zip(bpp_s, q_s, lambdas):
            ax.annotate(
                f"λ={int(lam)}", (x, y),
                textcoords="offset points", xytext=(4, 3), fontsize=7, color=COLORS["error_bars"], alpha=0.8,
            )


def _parse_quality_col(
    quality_col: "Union[str, Tuple[str, str]]",
) -> "Tuple[str, str]":
    """Return (col_key, y_label) from either a plain string or a (key, label) tuple."""
    if isinstance(quality_col, tuple):
        return quality_col[0], quality_col[1]
    return quality_col, quality_col.replace("_", " ")


def make_rd_figure(
    stats_df: pd.DataFrame,
    quality_col: "Union[str, Tuple[str, str]]" = "psnr_merlin",
    bpp_gpu_col: str = "bpp_likelihood",
    bpp_fpga_col: str = "bpp_bitstream",
    light_labels: bool = False,
    title_suffix: str = "",
    annotate_lambda: bool = False,
    save_path: Optional[Path] = None,
    ax: Optional[plt.Axes] = None,
) -> plt.Figure:
    """Full RD-curve figure: one curve per (arch, backend) combination.

    Color encodes architecture (ResSHyp=blue, SHyp=orange).
    Linestyle encodes backend (GPU=solid, FPGA=dashed).
    Error bands = ±1 std across seeds. Error bars = ±1 std on BPP.

    Args:
        quality_col: Either a plain string column key (e.g. ``"psnr_merlin"``) or a
                     ``(col_key, y_label)`` tuple to set the Y-axis label explicitly.
        ax:          If provided, the curve is drawn on that existing Axes object (for use
                     inside multi-metric grids). No figure is created, tight_layout() is
                     skipped, and save_path is ignored — the caller manages all of those.
    """
    col_key, col_label = _parse_quality_col(quality_col)

    _own_fig = ax is None
    if _own_fig:
        fig, ax = plt.subplots(figsize=(9, 6))
    else:
        fig = ax.figure

    # Discover all (arch, backend) combinations present in stats_df.
    # Sort so same-arch curves appear together; GPU solid drawn last (on top of FPGA dashed).
    unique_combos = sorted(set(
        (arch, backend)
        for _, backend, arch in stats_df.index
    ))

    for arch, backend in unique_combos:
        bpp_col = bpp_fpga_col if backend == "fpga" else bpp_gpu_col
        try:
            # Drop (backend, arch) levels from the index, leaving only lambda.
            backend_arch_stats = stats_df.xs((backend, arch), level=("backend", "arch"))
        except KeyError:
            continue
        lbl = (
            f"{arch} {backend.upper()}" if light_labels
            else f"{arch} {backend.upper()} ({'bitstream' if 'bitstream' in bpp_col else 'likelihood'})"
        )
        plot_rd_curve(
            ax, backend_arch_stats, backend, arch, bpp_col, col_key,
            label=lbl,
            annotate_lambda=annotate_lambda,
        )

    bpp_note = "bpp = likelihood (GPU) / rANS (FPGA)" if "likelihood" in bpp_gpu_col else "bpp = rANS bitstream"
    xlabel = "Bit-rate [bpp]" if light_labels else f"Bit-rate [bpp]  —  {bpp_note}"
    ax.set_xlabel(xlabel, fontsize=8 if not _own_fig else 10)
    ax.set_ylabel(col_label, fontsize=8 if not _own_fig else 10)
    if not light_labels:
        ax.set_title(
            f"{col_label}  —  {title_suffix}".strip(" —") if title_suffix else col_label,
            fontsize=8.5 if not _own_fig else 11,
        )
    ax.legend(loc="lower right", fontsize=7 if not _own_fig else 9)
    ax.grid(alpha=0.3)
    if _own_fig:
        plt.tight_layout()
        if save_path:
            fig.savefig(save_path, dpi=150)
            print(f"Saved: {save_path}")
    return fig


# ── Main plot: PSNR vs MERLIN ─────────────────────────────────────────────────
fig_merlin = make_rd_figure(
    stats_df,
    # quality_col=("psnr_merlin", "PSNR [dB], GPU vs FPGA"),
    quality_col=("epd_merlin", "EPD MERLIN, GPU vs FPGA"),
    bpp_gpu_col=BPP_GPU_COL,
    annotate_lambda=True,
)
plt.show()


### 6.1 Generating Paper figures

In [ ]:
fig_adam = make_rd_figure(
    stats_df,
    quality_col=("psnr_merlin", "PSNR [dB]"),
    # quality_col=("ssim_merlin", "SSIM"),
    bpp_gpu_col=BPP_GPU_COL,
    annotate_lambda=True,
    light_labels=True,
)
plt.show()


## 7 · FPGA degradation Δ plot

The key quantization question: **how much PSNR does the FPGA lose vs the GPU model?**

$$\Delta \text{PSNR}(\lambda) = \text{PSNR}_\text{FPGA}(\lambda) - \text{PSNR}_\text{GPU}(\lambda)$$

Computed per seed, then aggregated. A negative value means the FPGA is worse.
The error bands show the seed-to-seed variability of that degradation.

Note: when `BPP_GPU_COL == "bpp_likelihood"` the X-axis still shows GPU likelihood BPP
(since both backends are evaluated at the same λ, the X-axis here is just λ as a proxy).


In [ ]:

def compute_delta(df: pd.DataFrame, quality_col: str = "psnr_merlin") -> pd.DataFrame:
    """Compute per-seed FPGA - GPU delta for a quality metric, then aggregate.

    Uses pivot_table to align GPU and FPGA by (lambda, seed), then subtracts.
    Returns a DataFrame indexed by lambda with columns (delta, mean/std/min/max).
    """
    # pivot_table: rows=lambda×seed, columns=backend, values=quality_col
    # This gives one GPU value and one FPGA value per (lambda, seed) pair.
    pivot = df.pivot_table(index=["lambda", "seed"], columns="backend", values=quality_col)
    pivot = pivot.dropna(subset=["gpu", "fpga"])      # only pairs where both exist
    pivot["delta"] = pivot["fpga"] - pivot["gpu"]

    # Aggregate the delta across seeds for each lambda
    return (
        pivot["delta"]
        .groupby("lambda")
        .agg(["mean", "std", "min", "max"])
        .sort_index()
    )


def plot_delta(
    deltas: "List[Tuple[pd.DataFrame, str]]",
    bpp_means: pd.Series,
    arch: str,
    yaxis_label: Optional[str] = None,
    save_path: Optional[Path] = None,
) -> plt.Figure:
    """Bar chart of mean FPGA degradation per λ for one or more quality metrics.

    Args:
        deltas:    List of ``(delta_df, label)`` tuples.  Each *delta_df* is the
                   output of ``compute_delta()`` — a DataFrame indexed by λ with
                   columns ``mean / std / min / max``.
                   A single-element list preserves the original red/green-per-sign
                   colouring.  Multiple metrics produce a grouped bar chart with a
                   legend, one colour per metric.
        bpp_means: Mean GPU BPP per λ (used for X-tick labels).
        save_path: If given, save the figure at 150 dpi.
    """
    n_metrics = len(deltas)
    all_lambdas = sorted(set().union(*[set(d.index) for d, _ in deltas]))
    lambdas = np.array(all_lambdas, dtype=float)
    x = np.arange(len(lambdas))

    # Distribute n_metrics bars across a fixed group width of 0.70
    group_w = 0.70
    width   = group_w / n_metrics
    offsets = np.linspace(-group_w / 2 + width / 2, group_w / 2 - width / 2, n_metrics)

    metric_colors = [COLORS["psnr"], COLORS["ssim"], COLORS["epd"]]

    fig, ax = plt.subplots(figsize=(max(10, 1.5 * len(lambdas)), 4))

    for i, (delta, label) in enumerate(deltas):
        means = np.array([delta.loc[l, "mean"] if l in delta.index else float("nan") for l in lambdas])
        mins  = np.array([delta.loc[l, "min"]  if l in delta.index else float("nan") for l in lambdas])
        maxs  = np.array([delta.loc[l, "max"]  if l in delta.index else float("nan") for l in lambdas])

        pos = x + offsets[i]

        if n_metrics == 1:
            # Use the fpga color for bars where mean > 0 (FPGA better than GPU) and gpu color where mean < 0 (FPGA worse than GPU)
            # bar_colors = [COLORS["gpu"] if m < 0 else COLORS["fpga"] for m in means]
            bar_colors = COLORS["fpga"]
        else:
            bar_colors = metric_colors[i % len(metric_colors)]

        ax.bar(pos, means, width, color=bar_colors, alpha=0.75, label=label)
        ax.errorbar(
            pos, means,
            yerr=[np.nan_to_num(means - mins), np.nan_to_num(maxs - means)],
            fmt="none", color=COLORS["error_bars"], capsize=3, alpha=0.7,
        )

    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"λ={int(l)}\n({bpp_means.get(l, float('nan')):.3f} bpp)" for l in lambdas],
        fontsize=9,
    )
    ax.set_ylabel(yaxis_label or "Difference FPGA - GPU")
    if yaxis_label is None:
        ax.set_title(f"FPGA quantization degradation  —  {arch}\n(negative = FPGA worse than GPU)")
    ax.grid(axis="y", alpha=0.3)
    if n_metrics > 1:
        ax.legend(fontsize=8)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
        print(f"Saved: {save_path}")
    return fig


# ── Delta metrics to compare (col_key, display_label) ─────────────────────────
DELTA_METRICS: List[Tuple[str, str]] = [
    ("psnr_merlin", "PSNR [dB]"),
    ("ssim_merlin", "SSIM"),
    ("epd_merlin",   "EPD"),
]

ARCH = "ResSHyp"

deltas = [(compute_delta(df.query(f"arch == '{ARCH}'"), col), label) for col, label in DELTA_METRICS]

# Use GPU mean BPP as X-axis label (best available approximation per λ)
gpu_bpp_by_lambda = df.query(f"arch == '{ARCH}' and backend == 'gpu'").groupby("lambda")[BPP_GPU_COL].mean()

fig_delta = plot_delta(deltas, gpu_bpp_by_lambda, arch=ARCH, yaxis_label=f"Quality difference: FPGA - GPU for {ARCH}")
plt.show()

# # ── Per-metric summary table ───────────────────────────────────────────────────
# for (col, label), (delta, _) in zip(DELTA_METRICS, deltas):
#     print(f"\nMean FPGA degradation ({label}) across seeds:")
#     print(delta[["mean", "std", "min", "max"]].to_string(float_format="{:.3f}".format))
    
# Print average degradation for each metrics across lamdbas
print(f"\nAverage FPGA degradation across all λ values:")
for (col, label), (delta, _) in zip(DELTA_METRICS, deltas):
    avg_mean = delta["mean"].mean()
    print(f"  {label}: {avg_mean:.3f}")

if SAVE_FIGURES:
    output_path_delta = PLOTS_DIR / "FPGA_quantization_degradation.pdf"
    fig_delta.savefig(output_path_delta, dpi=150)
    print(f"Saved: {output_path_delta}")

### 7.1 Generating paper figures

In [ ]:
deltas = [(compute_delta(df.query(f"arch == '{ARCH}'"), "psnr_merlin"), "PSNR [dB]")]

# Use GPU mean BPP as X-axis label (best available approximation per λ)
gpu_bpp_by_lambda = df.query(f"arch == '{ARCH}' and backend == 'gpu'").groupby("lambda")[BPP_GPU_COL].mean()

fig_delta = plot_delta(deltas, gpu_bpp_by_lambda, arch=ARCH, yaxis_label=f"{deltas[0][1]} Difference FPGA - GPU for {ARCH}")
plt.show()

## 8 · BPP comparison — likelihood vs rANS bitstream

The GPU trains with a soft entropy estimate (likelihood BPP from the entropy bottleneck).
The FPGA uses real rANS coding. This plot shows how they compare at each λ.

- **If `update_wandb_runs.py` has run and added `bpp_bitstream` to W&B**: both backends
  show hard BPP and the comparison is apples-to-apples.
- **Currently**: GPU shows likelihood BPP (soft, usually slightly lower than real coding)
  and FPGA shows rANS BPP. The gap is the coding overhead of the real entropy coder.

> To add `bpp_bitstream` to the GPU runs: complete `update_wandb_runs.py` and re-run cell 2.
> `BPP_GPU_COL` will automatically switch to `"bpp_bitstream"`.


In [ ]:
def plot_bpp_comparison(
    df: pd.DataFrame,
    save_path: Optional[Path] = None,
) -> plt.Figure:
    """Side-by-side grouped bar chart: GPU likelihood BPP vs FPGA rANS BPP per λ.

    Even when BPP_GPU_COL == "bpp_bitstream" this cell is useful: it compares the
    two backends' real BPP values to show any coding overhead difference.
    """
    # Aggregate BPP per (lambda, backend) — mean and std across seeds
    bpp_stats = df.groupby(["lambda", "backend"]).agg(
        bpp_gpu_lik_mean=("bpp_likelihood", "mean"),
        bpp_gpu_lik_std=("bpp_likelihood", "std"),
        bpp_bitstream_mean=("bpp_bitstream", "mean"),
        bpp_bitstream_std=("bpp_bitstream", "std"),
    )

    lambdas = sorted(df["lambda"].unique())
    x = np.arange(len(lambdas))
    width = 0.25
    fig, ax = plt.subplots(figsize=(12, 5))

    gpu_lik_means, gpu_lik_stds = [], []
    gpu_bit_means, gpu_bit_stds = [], []
    fpga_means,    fpga_stds    = [], []

    for lam in lambdas:
        try:
            g_row = bpp_stats.loc[(lam, "gpu")]
            gpu_lik_means.append(g_row.get("bpp_gpu_lik_mean", float("nan")))
            gpu_lik_stds.append( g_row.get("bpp_gpu_lik_std",  0) or 0)
            gpu_bit_means.append(g_row.get("bpp_bitstream_mean", float("nan")))
            gpu_bit_stds.append( g_row.get("bpp_bitstream_std",  0) or 0)
        except KeyError:
            gpu_lik_means.append(float("nan")); gpu_lik_stds.append(0)
            gpu_bit_means.append(float("nan")); gpu_bit_stds.append(0)
        try:
            f_row = bpp_stats.loc[(lam, "fpga")]
            fpga_means.append(f_row.get("bpp_bitstream_mean", float("nan")))
            fpga_stds.append( f_row.get("bpp_bitstream_std",  0) or 0)
        except KeyError:
            fpga_means.append(float("nan")); fpga_stds.append(0)

    ax.bar(x - width, gpu_lik_means,  width, label="GPU likelihood BPP",  color=C["platforms"]["gpu_idle"],    yerr=gpu_lik_stds,  capsize=2)
    ax.bar(x,         gpu_bit_means,  width, label="GPU bitstream BPP",   color=C["platforms"]["gpu_dynamic"],  yerr=gpu_bit_stds,  capsize=2)
    ax.bar(x + width, fpga_means,     width, label="FPGA rANS BPP",       color=C["platforms"]["fpga_dynamic"], yerr=fpga_stds,     capsize=2)

    ax.set_xticks(x)
    ax.set_xticklabels([f"λ={int(l)}" for l in lambdas], fontsize=8)
    ax.set_ylabel("Bit-rate [bpp]")
    ax.set_title("BPP comparison: GPU likelihood / GPU bitstream / FPGA rANS  —  ResSHyp")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
        print(f"Saved: {save_path}")
    return fig


ARCH = "ResSHyp"
fig_bpp = plot_bpp_comparison(df.query(f"arch == '{ARCH}'"))
plt.show()

if SAVE_FIGURES:
    output_path_bpp = PLOTS_DIR / f"BPP_comparison_{ARCH}.pdf"
    fig_bpp.savefig(output_path_bpp, dpi=150)
    print(f"Saved: {output_path_bpp}")


## 9 · Hamburg Tile Visualization

Visual comparison of the reconstructions on the **1024 × 1024 Hamburg tile** for GPU and FPGA
across all (or a user-selected subset of) λ values.

**Layout** — 3 rows × N columns (N = max(n\_refs, n\_lambdas)):
- **Row 0 — References**: Noisy, MERLIN; optionally ADAM-NOC and MERLIN-DDS (set flags below).
- **Row 1 — FPGA**: one subplot per λ. BPP from `Hamburg_*_metrics.json` (already saved by
  `inference_hybrid.py`).
- **Row 2 — GPU**: one subplot per λ. BPP from `recon_*_metrics.json` (saved by
  `CompareReconstructionToGT.on_test_end()` after re-running `update_wandb_runs.py`); shows `N/A`
  until that file exists.

All images are log-I, clipped to the **5th–95th percentile of the Noisy tile** (same reference
for every subplot → consistent brightness scale across the whole figure).

**PSNR note** — recomputed here on-the-fly from the `.npy` arrays using the same clip-first
formula as `inference_utils.MetricsTracker.compute_psnr` (clip both arrays to `AMP_LIN_99`
before MSE). This avoids inflated MSE from occasional GPU model output outliers.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ── Configuration — edit these to change what is displayed ───────────────────
# ─────────────────────────────────────────────────────────────────────────────
TILE_NAME    = "Hamburg_[11000:12024-8500:9524]"
SHOW_LAMBDAS = [1, 5, 20, 200, 1000]      # None → all 10 lambdas; else list, e.g., [1, 20, 50, 200, 1000]
SEED_FOR_VIS = 1         # which seed's images to display
ARCH_FOR_VIS = "ResSHyp" # which architecture to display in the tile visualization

SHOW_ADAM_NOC   = True  # set True to add ADAM-NOC column in reference row
SHOW_MERLIN_DDS = True  # set True to add MERLIN-DDS column in reference row

# SHOW_ENL_ROI    = True  # set True to overlay the ENL ROI rectangle on the Noisy panel
# # ENL ROI: rows 800-1000, cols 400-600 (water body — homogeneous region)
# _ENL_ROI_R0, _ENL_ROI_R1, _ENL_ROI_C0, _ENL_ROI_C1 = 400, 600, 800, 1000
SHOW_CLOSEUP_ROI = True  # set True to add a close-up crop of the tile (next figure) — edit the CROP_R/C values below to move/resize
CROP_R0, CROP_R1, CROP_C0, CROP_C1 = 0, 200, 100, 300

REFS_DIR = ROOT_DIR / "data" / "visualization" / TILE_NAME


# ─────────────────────────────────────────────────────────────────────────────
# ── Helpers ───────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────
def _linA_to_logI(linA: np.ndarray) -> np.ndarray:
    """Linear amplitude → log-intensity (same formula used throughout the project)."""
    return np.log(np.square(linA) + EPS)


def _compute_psnr_tile(recon: np.ndarray, ref: np.ndarray) -> float:
    """PSNR using src.utils.metrics.psnr — mse() already clips both to AMP_LIN_99 first."""
    r = torch.from_numpy(recon.astype(np.float32))
    t = torch.from_numpy(ref.astype(np.float32))
    return _src_psnr(r, t)


# ─────────────────────────────────────────────────────────────────────────────
# ── Load reference images ─────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────
noisy_linA  = np.load(REFS_DIR / "linA_Noisy.npy")
merlin_linA = np.load(REFS_DIR / "linA_MERLIN.npy")
adam_noc_linA   = np.load(REFS_DIR / "linA_ADAM_NOC.npy")   if SHOW_ADAM_NOC   else None
merlin_dds_linA = np.load(REFS_DIR / "linA_MERLIN_DDS.npy") if SHOW_MERLIN_DDS else None

# ─────────────────────────────────────────────────────────────────────────────
# ── Load FPGA & GPU reconstructions for selected lambdas ─────────────────────
# ─────────────────────────────────────────────────────────────────────────────
sel_lambdas: List[float] = sorted(SHOW_LAMBDAS) if SHOW_LAMBDAS else sorted(df["lambda"].unique())

# tile_data[lmbda][backend] = {"linA", "logI", "psnr_merlin", "bpp"} or None if missing
tile_data: Dict[float, Dict[str, Optional[Dict]]] = {}

for lmbda in sel_lambdas:
    tile_data[lmbda] = {}
    for backend in ["fpga", "gpu"]:
        rows = df[
            (df["lambda"] == lmbda)
            & (df["seed"]   == SEED_FOR_VIS)
            & (df["backend"] == backend)
            & (df["arch"]   == ARCH_FOR_VIS)
        ]
        if len(rows) == 0:
            tile_data[lmbda][backend] = None
            continue
        row = rows.iloc[0]

        # Paths differ per backend; columns run_dir / model_dir are NaN for the other backend
        if backend == "fpga":
            npy_path     = Path(row["model_dir"]) / "results" / f"{TILE_NAME}_recon_linA.npy"
            metrics_path = Path(row["model_dir"]) / "results" / f"{TILE_NAME}_metrics.json"
        else:
            npy_path     = Path(row["run_dir"]) / f"recon_{TILE_NAME}_linA.npy"
            metrics_path = Path(row["run_dir"]) / f"recon_{TILE_NAME}_metrics.json"

        if not npy_path.exists():
            print(f"  {y}MISSING{e}: λ={int(lmbda)} seed={SEED_FOR_VIS} {backend} — {npy_path.name}")
            tile_data[lmbda][backend] = None
            continue

        linA = np.load(npy_path)
        psnr_vs_merlin = _compute_psnr_tile(linA, merlin_linA)

        # BPP from per-tile metrics JSON: "bpp" for FPGA, "bpp_bitstream" for GPU
        bpp = float("nan")
        if metrics_path.exists():
            m = json.loads(metrics_path.read_text())
            bpp_key = "bpp" if backend == "fpga" else "bpp_bitstream"
            bpp = float(m.get(bpp_key, float("nan")))

        tile_data[lmbda][backend] = {
            "linA":        linA,
            "logI":        _linA_to_logI(linA),
            "psnr_merlin": psnr_vs_merlin,
            "bpp":         bpp,
        }

n_present = sum(1 for d in tile_data.values() for v in d.values() if v is not None)
print(f"Loaded {n_present} / {2 * len(sel_lambdas)} tiles  (seed={SEED_FOR_VIS}, {len(sel_lambdas)} λ × 2 backends)")

# ─────────────────────────────────────────────────────────────────────────────
# ── Plot ─────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────
ref_panels: List[Tuple[str, np.ndarray]] = [("Noisy", noisy_linA), ("MERLIN", merlin_linA)]
if adam_noc_linA   is not None: ref_panels.append(("ADAM-NOC",   adam_noc_linA))
if merlin_dds_linA is not None: ref_panels.append(("MERLIN-DDS", merlin_dds_linA))

n_refs = len(ref_panels)
n_lam  = len(sel_lambdas)
n_cols = max(n_refs, n_lam)
n_rows = 3
SUBPLOT_IN = 2.6   # inches per subplot

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(SUBPLOT_IN * n_cols + 0.5, SUBPLOT_IN * n_rows + 0.7),
    squeeze=False,
)

# ── Row header annotations (text on transAxes, visible even after axis("off")) ──
_ROW_HEADERS = [
    (0, "References",                "black"),
    (1, f"FPGA\n(seed={SEED_FOR_VIS})", COLORS["fpga"]),
    (2, f"GPU\n(seed={SEED_FOR_VIS})",  COLORS["gpu"]),
]
for ri, label, color in _ROW_HEADERS:
    axes[ri, 0].text(
        -0.12, 0.5, label,
        transform=axes[ri, 0].transAxes,
        rotation=90, va="center", ha="right",
        fontsize=9, fontweight="bold", color=color,
    )


def _show(ax: plt.Axes, logI: np.ndarray, title: str,
          subtitle: Optional[str] = None, border_color: Optional[str] = None) -> None:
    """Display a clipped log-I image with optional border + subtitle.

    Clipping: mean ± 3·std applied per-image (mirrors CompareReconstructionToGT callback).
    """
    clipped = clip_logI(logI, mean_std_norm=True, clip_factor=3)
    ax.imshow(clipped, cmap="gray", aspect="equal", interpolation="none")
    ax.set_title(title, fontsize=8, fontweight="bold", pad=3)
    if subtitle:
        ax.text(0.5, -0.03, subtitle, transform=ax.transAxes,
                fontsize=6.5, ha="center", va="top", color="dimgray")
    ax.axis("off")
    if border_color:
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor(border_color)
            spine.set_linewidth(2.0)


# ── Row 0: References ─────────────────────────────────────────────────────────
for ci, (ref_name, ref_linA) in enumerate(ref_panels):
    _show(axes[0, ci], _linA_to_logI(ref_linA), ref_name)
    if SHOW_CLOSEUP_ROI and ref_name == "Noisy":
        roi_rect = Rectangle(
            (CROP_C0, CROP_R0),               # (x=col, y=row)
            CROP_C1 - CROP_C0,                # width
            CROP_R1 - CROP_R0,                # height
            linewidth=1.5, edgecolor="red", facecolor="none", linestyle="--",
        )
        axes[0, ci].add_patch(roi_rect)
    # Overlay ENL ROI rectangle on the Noisy panel to show the homogeneous evaluation region
    # if SHOW_ENL_ROI and ref_name == "Noisy":
    #     roi_rect = Rectangle(
    #         (_ENL_ROI_C0, _ENL_ROI_R0),               # (x=col, y=row)
    #         _ENL_ROI_C1 - _ENL_ROI_C0,                # width
    #         _ENL_ROI_R1 - _ENL_ROI_R0,                # height
    #         linewidth=1.5, edgecolor="red", facecolor="none", linestyle="--",
    #     )
    #     axes[0, ci].add_patch(roi_rect)
    #     axes[0, ci].text(
    #         _ENL_ROI_C0 + 2, _ENL_ROI_R0 - 4, "ENL ROI",
    #         color="red", fontsize=5.5, va="bottom",
    #     )
for ci in range(n_refs, n_cols):
    axes[0, ci].axis("off")

# ── Rows 1 (FPGA) & 2 (GPU) ──────────────────────────────────────────────────
for row_idx, backend in enumerate(["fpga", "gpu"], start=1):
    color = COLORS[backend]
    for ci, lmbda in enumerate(sel_lambdas):
        data = tile_data.get(lmbda, {}).get(backend)
        if data is None:
            axes[row_idx, ci].axis("off")
            axes[row_idx, ci].text(0.5, 0.5, "N/A", ha="center", va="center",
                                   transform=axes[row_idx, ci].transAxes, fontsize=10, color="gray")
            continue
        bpp_str  = f"{data['bpp']:.3f}" if not np.isnan(data["bpp"]) else "N/A"
        subtitle = f"BPP={bpp_str}  PSNR={data['psnr_merlin']:.2f}dB"
        _show(axes[row_idx, ci], data["logI"], f"λ={int(lmbda)}", subtitle, border_color=color)
    for ci in range(n_lam, n_cols):
        axes[row_idx, ci].axis("off")

fig.suptitle(
    f"Hamburg Tile  —  {TILE_NAME}\n"
    f"log-I · per-image mean±3σ clipping  |  PSNR vs MERLIN  |  seed={SEED_FOR_VIS}",
    fontsize=10, y=1.01,
)
plt.tight_layout(h_pad=1.2, w_pad=0.3)
plt.show()
# save image as pdf in the same directory as the notebook
if SAVE_FIGURES:
    output_path = PLOTS_DIR / "Visualizations_Hamburg.pdf"
    fig.savefig(output_path, dpi=150)
    print(f"Saved: {output_path}")


## 9-bis · Hamburg Tile — Close-up

In [ ]:
def _crop(arr: np.ndarray) -> np.ndarray:
    return arr[CROP_R0:CROP_R1, CROP_C0:CROP_C1]


n_refs_cu = len(ref_panels)
n_lam_cu  = len(sel_lambdas)
n_cols_cu = max(n_refs_cu, n_lam_cu)
n_rows_cu = 3

fig_cu, axes_cu = plt.subplots(
    n_rows_cu, n_cols_cu,
    figsize=(SUBPLOT_IN * n_cols_cu + 0.5, SUBPLOT_IN * n_rows_cu + 0.7),
    squeeze=False,
)

# Row header annotations
for ri, label, color in _ROW_HEADERS:
    axes_cu[ri, 0].text(
        -0.12, 0.5, label,
        transform=axes_cu[ri, 0].transAxes,
        rotation=90, va="center", ha="right",
        fontsize=9, fontweight="bold", color=color,
    )

# Row 0: References (cropped)
for ci, (ref_name, ref_linA) in enumerate(ref_panels):
    _show(axes_cu[0, ci], _crop(_linA_to_logI(ref_linA)), ref_name)
for ci in range(n_refs_cu, n_cols_cu):
    axes_cu[0, ci].axis("off")

# Rows 1 (FPGA) & 2 (GPU) — cropped
for row_idx, backend in enumerate(["fpga", "gpu"], start=1):
    color = COLORS[backend]
    for ci, lmbda in enumerate(sel_lambdas):
        data = tile_data.get(lmbda, {}).get(backend)
        if data is None:
            axes_cu[row_idx, ci].axis("off")
            axes_cu[row_idx, ci].text(0.5, 0.5, "N/A", ha="center", va="center",
                                      transform=axes_cu[row_idx, ci].transAxes,
                                      fontsize=10, color="gray")
            continue
        bpp_str  = f"{data['bpp']:.3f}" if not np.isnan(data["bpp"]) else "N/A"
        subtitle = f"BPP={bpp_str}  PSNR={data['psnr_merlin']:.2f}dB"
        _show(axes_cu[row_idx, ci], _crop(data["logI"]), f"λ={int(lmbda)}", subtitle, border_color=color)
    for ci in range(n_lam_cu, n_cols_cu):
        axes_cu[row_idx, ci].axis("off")

fig_cu.suptitle(
    f"Hamburg Tile — Close-up  rows [{CROP_R0}:{CROP_R1}]  cols [{CROP_C0}:{CROP_C1}]\n"
    f"log-I · per-image mean±3σ clipping  |  PSNR vs MERLIN  |  seed={SEED_FOR_VIS}",
    fontsize=10, y=1.01,
)
plt.tight_layout(h_pad=1.2, w_pad=0.3)
plt.show()
if SAVE_FIGURES:
    output_path_cu = PLOTS_DIR / "Visualizations_Hamburg_closeup.pdf"
    fig_cu.savefig(output_path_cu, dpi=150)
    print(f"Saved: {output_path_cu}")


### Other plotting tests

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ── SAR Quality Metric RD-curve  —  switchable Y axis ────────────────────────
# ─────────────────────────────────────────────────────────────────────────────
# Switch QUALITY_COL to any column in METRIC_COLS to update the Y axis.
# Examples:
#   "enl_recon"    → Equivalent Number of Looks on the reconstruction (higher = less speckle)
#   "ratio_mean"   → Mean of noisy_I / recon_I ratio (1.0 = ideal; >1 = residual speckle)
#   "ratio_enl"    → ENL of the ratio image (should be ~ L for a well-calibrated filter)
#   "epd_merlin"   → Edge Preservation Degree vs MERLIN (1.0 = perfect)
#   "epd_adam_noc" → Edge Preservation Degree vs ADAM-NOC (1.0 = perfect)
#   "psnr_merlin"  → back to classic PSNR

QUALITY_COL = "ratio_enl"

# Human-readable Y-axis labels per metric.
# Comment out any entry to exclude it from the multi-metric grid below.
_METRIC_LABELS: Dict[str, str] = {
    "psnr_merlin":  "PSNR vs MERLIN [dB]",
    "ssim_merlin":  "SSIM vs MERLIN",
    "epd_merlin":   "EPD vs MERLIN", #  (1.0 = perfect edge preservation)",
    # "mse_merlin":   "MSE vs MERLIN",
    "ratio_mean":   "Mean of Ratio image (noisy_I / recon_I)", #  (1.0 = ideal)",
    "ratio_enl":    "ENL of Ratio image", #  (higher = well-calibrated)",
    "enl_recon":    "ENL of Reconstruction (whole patches)", #  (higher = smoother)",
}
    # "psnr_adam_noc":"PSNR vs ADAM-NOC [dB]",
    # "epd_adam_noc": "EPD vs ADAM-NOC  (1.0 = perfect edge preservation)",

# ────────────────────────────────────────────────────────────────────────
# # ───────── One large visualization of a specific quality metric ─────────
# # ────────────────────────────────────────────────────────────────────────
# fig_sar = make_rd_figure(
#     stats_df,
#     quality_col=(QUALITY_COL, _METRIC_LABELS.get(QUALITY_COL, QUALITY_COL)),
#     bpp_gpu_col=BPP_GPU_COL,
#     annotate_lambda=True,
# )
# if SAVE_FIGURES:
#     fig_sar.savefig(PLOTS_DIR / f"rd_{QUALITY_COL}.pdf", dpi=150, bbox_inches="tight")
# plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# ── Multi-metric RD-curve grid  —  one subplot per entry in _METRIC_LABELS ───
# ─────────────────────────────────────────────────────────────────────────────
# Comment out entries in _METRIC_LABELS above to remove them from this grid.

MULTI_METRIC_TITLE = "GPU vs FPGA RD-curves  —  ResSHyp-relu  —  All Quality Metrics"
N_COLS_GRID = 3  # 2 or 3 — figure grows taller rather than wider

n_metrics   = len(_METRIC_LABELS)
n_rows_grid = (n_metrics + N_COLS_GRID - 1) // N_COLS_GRID

fig_all, axes_all = plt.subplots(
    n_rows_grid, N_COLS_GRID,
    figsize=(N_COLS_GRID * 5.5, n_rows_grid * 4.2),
    squeeze=False,
)

for idx, (metric_col, metric_label) in enumerate(_METRIC_LABELS.items()):
    _r, _c = divmod(idx, N_COLS_GRID)
    make_rd_figure(
        stats_df,
        quality_col=(metric_col, metric_label),
        bpp_gpu_col=BPP_GPU_COL,
        annotate_lambda=False,
        ax=axes_all[_r, _c],
    )

# Hide unused subplots (when n_metrics is not a multiple of N_COLS_GRID)
for idx in range(n_metrics, n_rows_grid * N_COLS_GRID):
    _r, _c = divmod(idx, N_COLS_GRID)
    axes_all[_r, _c].axis("off")

fig_all.suptitle(MULTI_METRIC_TITLE, fontsize=12, fontweight="bold")
plt.tight_layout()

if SAVE_FIGURES:
    fig_all.savefig(PLOTS_DIR / "rd_all_metrics.pdf", dpi=150, bbox_inches="tight")
    print(f"Saved: {PLOTS_DIR / 'rd_all_metrics.pdf'}")

plt.show()
